# Preliminary data exploration

The file \src\data_extraction\extract_cities.py analyses the raw dataset by loading the original csv files into temporary DuckDB tables. These are then filtered for specific cities (defined inside extract_cities.py) and the corresponding compressed parquet files are built.

In order to inspect what the files contain let's first define the working path.

In [1]:
from pathlib import Path

cwd = Path.cwd()
for folder in (cwd, *cwd.parents):
    if (folder / "src" / "data_extraction" / "extract_cities.py").is_file():
        PROJECT_ROOT = folder
        break
else:
    raise RuntimeError("Start Jupyter from the CityFlow folder or one of its subfolders.")

For each city we have the following files:

1) **cities** is single-row df containing : city id, name, coordinates (latitude and longitude), timezone, country and return_to_official_only (True = bikes can only be returned to official dock stations, False = bikes can be returned everywhere in designated zones)
2) **station_status** is a df containing : station_id, time, total numb of bikes, booked bikes, bikes available to rent, free racks, free special racks, maintenance status
3) **stations** is a df containing : id (equivalent to station_id), city_id (identical for each row), name, app_number, **terminal_type (not sure what this represents**), place_type, # of bike_racks, # of special_racks, rack_locks and coordinates (latitude and longitude)
4) **trips** is a df containing : bike_id, city_id (identical for each row), time_start, coordinates_start (latitude and longitude), coordinates_end (latitude and longitude), station_id_start, station_id_end, battery_start, battery_end, duration (seconds), distance (meters)

In [24]:
import duckdb

path = PROJECT_ROOT / "data" / "heidelberg" / "trips.parquet"
duckdb.read_parquet(str(path)).limit(5).df()

,bike_id,city_id,time_start,lon_start,lat_start,lon_end,lat_end,station_id_start,station_id_end,battery_start,battery_end,duration,distance
0,42763,194,1.685449e+09,8.671190,49.416215,8.693737,49.408694,1250591,59656724,None,None,660.0,1833.044568
1,42763,194,1.685453e+09,8.693737,49.408694,8.706719,49.411060,59656724,121361,None,None,780.0,975.377219
2,42763,194,1.685463e+09,8.706719,49.411060,8.687870,49.407324,121361,<NA>,None,None,480.0,1425.584229
3,42763,194,1.685531e+09,8.687991,49.408170,8.706719,49.411060,661043,121361,None,None,600.0,1392.533391
4,42763,194,1.685547e+09,8.706719,49.411060,8.666115,49.377529,121361,339522,None,None,1440.0,4747.287396


**Initially, I would focus on a city where return_to_official_only=True. This way the position of available bikes should be well-defined at any moment.**

**Idea :** we could make a classification model that tells you whether a specific journey is possible in a fixed time (e.g. 2 hours from now). Based on start and end dock station coordinates, we should predict the probability that at least 2 bikes are available in station_start 2 hours from now, and the probability that at least 2 racks are free in station_end 2 hours + travel time from now. The avg travel time between two stations can be computed from trips.parquet.

## Inspecting Data Quality

**Check the return_to_official policy**

For cities where return_to_official_only is True, we would expect that each trip starts and ends in official dock stations. In practice, we find:
- Heidelberg : 12.9% (return_to_official_only = True)
- Bergamo :    12.9% (return_to_official_only = True)
- Karlsruhe :  97.1% (return_to_official_only = False) 

In [3]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))
from src.inspection.trips import count_trips_missing_station_ids

city_dir = PROJECT_ROOT / "data" / "karlsruhe"
trips_path = city_dir / "trips.parquet"

count_trips_missing_station_ids(city_dir) / duckdb.execute('SELECT COUNT(*) FROM read_parquet(?)', [str(trips_path)]).fetchall()[0][0]

0.970818118514241

The percentage, of failed return_to_official for Heidelberg and Bergamo are suspisciously close. Let's analyse if this problem arises uniformly during the year or whether they occour at specific dates.

In [28]:
from src.inspection.trips import days_with_missing_station_ids

city_dir = PROJECT_ROOT / "data" / "karlsruhe"
trips_path = city_dir / "trips.parquet"

days_with_missing_station_ids(city_dir, n=500)

([datetime.date(2022, 8, 26),
  datetime.date(2022, 8, 27),
  datetime.date(2022, 8, 28),
  datetime.date(2022, 8, 29),
  datetime.date(2022, 8, 30),
  datetime.date(2022, 8, 31),
  datetime.date(2022, 9, 1),
  datetime.date(2022, 9, 2),
  datetime.date(2022, 9, 3),
  datetime.date(2022, 9, 4),
  datetime.date(2022, 9, 5),
  datetime.date(2022, 9, 6),
  datetime.date(2022, 9, 7),
  datetime.date(2022, 9, 8),
  datetime.date(2022, 9, 9),
  datetime.date(2022, 9, 10),
  datetime.date(2022, 9, 11),
  datetime.date(2022, 9, 12),
  datetime.date(2022, 9, 13),
  datetime.date(2022, 9, 14),
  datetime.date(2022, 9, 15),
  datetime.date(2022, 9, 16),
  datetime.date(2022, 9, 17),
  datetime.date(2022, 9, 19),
  datetime.date(2022, 9, 20),
  datetime.date(2022, 9, 21),
  datetime.date(2022, 9, 22),
  datetime.date(2022, 9, 23),
  datetime.date(2022, 9, 24),
  datetime.date(2022, 9, 25),
  datetime.date(2022, 9, 26),
  datetime.date(2022, 9, 27),
  datetime.date(2022, 9, 28),
  datetime.date(202

**Check the bike usage during the year**

1) First method can count how many dock stations changes there are for each day in the spanned period